# Onshape REST API — Phase 0 spike

The Onshape counterpart of `../3dx_integration/3dx_rest_api_tour.ipynb`. It walks the
calls the catalog backend will need, against your real Onshape account, and is built to
settle the **verify list** in `onshape_catalog_integration_plan.md` (§13).

**Read this first: every successful call costs quota.** The lab's Educator plan has
**2,500 successful calls per year for the whole company**. This notebook is built around
that:

- **Responses are cached to disk** (`catalog/onshape_spike/`, gitignored). Re-running a
  cell replays the saved response and costs nothing. Set `REFRESH = True` only when you
  need fresh data.
- **A call ledger** records every live call. A hard cap (`SPIKE_CALL_BUDGET`, default
  150) refuses further live calls once reached, so a loop can't spend the year's quota.
- **4xx/5xx responses are free**, so the error-probing cells (bad signature, clock skew)
  cost nothing.
- Each section header gives its expected cost. A full first run is roughly **30–45
  calls**.

**Before running, set up Onshape:**

1. An API key pair: *My Account → Developer → API keys* (needs read + write scope for
   §9). Put it in `.env` as `ONSHAPE_ACCESS_KEY` / `ONSHAPE_SECRET_KEY`, or the
   notebook will prompt.
2. A folder (default name `Inspection Parts`) holding **one test document** with a Part
   Studio of **two simple parts** (call them A and B).
3. In that document, make the versions that the change-stamp experiment (§6) needs:
   - **V1**: both parts as they are.
   - **V2**: change *only part A's geometry* (e.g. extrude depth), then version.
   - **V3**: change nothing in the Part Studio. Add any non-geometry change (rename a
     tab, or add an empty Variable Studio), then version.

Run top to bottom. Sections 9 (writes) are off until you set `ALLOW_WRITES = True`.

---
## 1. Settings

In [ ]:
import os
from pathlib import Path

# --- Account / stack ----------------------------------------------------------
BASE_URL = os.environ.get("ONSHAPE_BASE_URL", "https://cad.onshape.com").rstrip("/")
# The OpenAPI spec lists v17 as current (Oct 2026). Pin per call if one misbehaves.
API_VERSION = os.environ.get("ONSHAPE_API_VERSION", "v17")

# --- Scope ----------------------------------------------------------------------
# The folder id, or its whole URL (...documents?nodeId=<id>&resourceType=folder).
# Search by folder name does not work (see §4).
SCOPE_FOLDER = os.environ.get("ONSHAPE_SCOPE_FOLDER", "Inspection Parts")

# --- Quota safety -----------------------------------------------------------------
SPIKE_CALL_BUDGET = int(os.environ.get("ONSHAPE_SPIKE_CALL_BUDGET", "150"))
REFRESH = False          # True = ignore cached GET responses for this run
ALLOW_WRITES = False     # §9 creates and deletes a tab in the test document

# --- Where cached responses, the ledger and downloaded files go ----------------------
# Run on the host: docs/ isn't mounted in the container. Find the inspection-docker root.
REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "docker" / "docker-compose.yaml").exists():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "docker" / "docker-compose.yaml").exists():
    raise RuntimeError("Run this notebook from inside the inspection-docker checkout.")
# ./catalog on the host is /catalog in the container, so files saved here are
# visible to the container too. (.env's CATALOG_ROOT=/catalog is the container path.)
CATALOG_ROOT = REPO_ROOT / "catalog"
SPIKE_DIR = CATALOG_ROOT / "onshape_spike"
(SPIKE_DIR / "responses").mkdir(parents=True, exist_ok=True)
(SPIKE_DIR / "files").mkdir(parents=True, exist_ok=True)

print(f"API      : {BASE_URL}/api/{API_VERSION}")
print(f"Scope    : {SCOPE_FOLDER!r}")
print(f"Cache    : {SPIKE_DIR}")
print(f"Budget   : {SPIKE_CALL_BUDGET} live successful calls")
print(f"Writes   : {'ENABLED' if ALLOW_WRITES else 'disabled'}")

In [ ]:
import getpass

ACCESS_KEY = os.environ.get("ONSHAPE_ACCESS_KEY") or getpass.getpass("Onshape access key: ")
SECRET_KEY = os.environ.get("ONSHAPE_SECRET_KEY") or getpass.getpass("Onshape secret key: ")
print(f"Keys captured (access key ends …{ACCESS_KEY[-4:]}). Not stored in this file.")

---
## 2. A signed, quota-aware client

Onshape API keys authenticate each request on its own. There's no login and no session
cookie: every request carries three headers, and the signature covers the method, the
path, the query and the content type. This is what replaces the whole CAS/CSRF dance
from 3DX.

```
S = lower("{METHOD}\n{nonce}\n{date}\n{content-type}\n{path}\n{query}\n")
Authorization: On {accessKey}:HmacSHA256:{base64(hmac_sha256(secret, S))}
Date:          RFC 1123, within 5 min of Onshape's clock
On-Nonce:      ≥16 random alphanumerics, never reused
```

Three rules that the client below enforces:

- **Sign exactly what is sent.** Build the query string ourselves and sign the prepared
  request, so `requests` can't re-encode anything after signing. Multipart uploads put
  a boundary in `Content-Type`; that full string is what gets signed.
- **Handle 307 redirects by hand.** Some exports and downloads redirect, and a redirect
  needs a new signature for the new path. `requests` would follow it with the old one.
- **Count and cache.** 2xx/3xx responses go in the ledger; GET responses are cached by
  method + path + query.

In [ ]:
import base64
import datetime as dt
import email.utils
import hashlib
import hmac
import json
import secrets
import re
import string
import time
import urllib.parse

import requests

LEDGER = SPIKE_DIR / "call_ledger.jsonl"
JSON_ACCEPT = "application/json;charset=UTF-8; qs=0.09"
_ALNUM = string.ascii_letters + string.digits
session = requests.Session()


class BudgetExceeded(RuntimeError):
    pass


def calls_used():
    """Successful (billable) live calls recorded so far, across kernel restarts."""
    if not LEDGER.exists():
        return 0
    with LEDGER.open() as f:
        return sum(1 for line in f if json.loads(line)["billable"])


def _record(method, path, status):
    billable = 200 <= status < 400
    with LEDGER.open("a") as f:
        f.write(json.dumps({"ts": dt.datetime.now(dt.timezone.utc).isoformat(),
                            "method": method, "path": path, "status": status,
                            "billable": billable}) + "\n")
    return billable


def _sign(prepared, date=None, nonce=None, secret=None):
    """Attach Onshape HMAC headers to a prepared request (in place)."""
    parsed = urllib.parse.urlsplit(prepared.url)
    date = date or email.utils.formatdate(usegmt=True)
    nonce = nonce or "".join(secrets.choice(_ALNUM) for _ in range(25))
    ctype = prepared.headers.get("Content-Type", "")
    msg = (f"{prepared.method}\n{nonce}\n{date}\n{ctype}\n"
           f"{parsed.path}\n{parsed.query}\n").lower()
    sig = base64.b64encode(hmac.new((secret or SECRET_KEY).encode(), msg.encode(),
                                    hashlib.sha256).digest()).decode()
    prepared.headers.update({
        "Date": date,
        "On-Nonce": nonce,
        "Authorization": f"On {ACCESS_KEY}:HmacSHA256:{sig}",
    })
    return prepared


def _url(path, params=None):
    """Absolute API URL. `path` is relative to /api/{version} unless it starts with http."""
    if path.startswith("http"):
        # Paging links ('next') come back without the version (…/api/documents?…);
        # put ours back so every page is served by the same API version.
        url = re.sub(r"/api/(?!v\d+/)", f"/api/{API_VERSION}/", path, count=1)
    else:
        url = f"{BASE_URL}/api/{API_VERSION}/{path.lstrip('/')}"
    if params:
        query = urllib.parse.urlencode(
            {k: v for k, v in params.items() if v is not None}, doseq=True)
        url += ("&" if "?" in url else "?") + query
    return url


def _cache_file(method, url):
    return SPIKE_DIR / "responses" / (hashlib.sha256(f"{method} {url}".encode()).hexdigest()[:20] + ".json")


class Cached:
    """Minimal stand-in for a requests.Response replayed from disk."""
    def __init__(self, blob):
        self.status_code = blob["status"]
        self.url = blob["url"]
        self.headers = blob["headers"]
        self._json = blob.get("json")
        # Only JSON bodies are cached; rebuild .text from them so callers that
        # check r.text (e.g. the §3.1 auth check) see the same thing as live.
        self.text = json.dumps(self._json) if self._json is not None else ""
        self.from_cache = True
        self.ok = 200 <= self.status_code < 400

    def json(self):
        if self._json is None:
            raise ValueError("cached response has no JSON body")
        return self._json


def os_request(method, path, params=None, json_body=None, files=None, data=None,
               accept=JSON_ACCEPT, cache=None, sign_kwargs=None, stream=False,
               max_redirects=3):
    """One signed Onshape call, with caching, ledger and manual 307 handling.

    cache defaults to True for GET and False otherwise. Binary bodies are not cached;
    callers save them to SPIKE_DIR/files themselves.
    """
    method = method.upper()
    url = _url(path, params)
    cache = (method == "GET") if cache is None else cache
    cfile = _cache_file(method, url)
    if cache and not REFRESH and cfile.exists():
        return Cached(json.loads(cfile.read_text()))

    if calls_used() >= SPIKE_CALL_BUDGET:
        raise BudgetExceeded(f"{SPIKE_CALL_BUDGET} billable calls used; raise "
                             "SPIKE_CALL_BUDGET deliberately if you mean to continue.")

    headers = {"Accept": accept}
    if json_body is not None:
        headers["Content-Type"] = "application/json"
    for _ in range(max_redirects + 1):
        req = requests.Request(method, url, headers=headers, json=json_body,
                               files=files, data=data)
        prepared = _sign(session.prepare_request(req), **(sign_kwargs or {}))
        # prepare_request merges session headers; re-sign is not needed since the
        # signed fields (method/path/query/content-type) are final at this point.
        response = session.send(prepared, allow_redirects=False, stream=stream,
                                timeout=60)
        _record(method, urllib.parse.urlsplit(url).path, response.status_code)
        if response.status_code in (301, 302, 303, 307, 308):
            location = urllib.parse.urljoin(url, response.headers["Location"])
            host = urllib.parse.urlsplit(location).hostname or ""
            if not host.endswith("onshape.com"):
                # Never send the API key to a non-Onshape host; pre-signed storage
                # URLs need no auth anyway.
                return session.get(location, stream=stream, timeout=60)
            url = location
            continue
        break
    response.from_cache = False

    if cache and response.ok and "json" in response.headers.get("Content-Type", ""):
        cfile.write_text(json.dumps({
            "status": response.status_code, "url": response.url,
            "headers": {k: v for k, v in response.headers.items()
                        if k.lower() in ("content-type", "date")},
            "json": response.json()}))
    return response


def os_get(path, **kw):
    return os_request("GET", path, **kw)


def show(response, label=None, limit=1500):
    """Print a response compactly: method, URL, status, source, body."""
    if label:
        print(f"### {label}")
    src = "cache" if getattr(response, "from_cache", False) else "live"
    print(f"{response.url}\n--> {response.status_code} ({src}, "
          f"{response.headers.get('Content-Type', '?')})")
    try:
        body = json.dumps(response.json(), indent=2)
    except ValueError:
        body = response.text
    print(body[:limit] + ("\n... [truncated]" if len(body) > limit else ""))
    print(f"[ledger: {calls_used()}/{SPIKE_CALL_BUDGET} billable]\n")


def brief(obj, limit=1500):
    text = json.dumps(obj, indent=2, default=str)
    print(text[:limit] + ("\n... [truncated]" if len(text) > limit else ""))


FINDINGS = {}   # filled in as we go; printed in §11

print(f"Client ready. Ledger so far: {calls_used()} billable calls.")

---
## 3. Authentication — *1–2 billable calls; error probes are free*

### 3.1 Who am I?

`GET /users/sessioninfo` returns who the key belongs to, the **scopes the key was
granted** (`oauth2Scopes`) and the **plan** (`companyPlan`, `planGroup`). Your user id
is needed for the company lookup.

Status codes seen so far, and what they mean:

| Response | Meaning |
|---|---|
| `200` with a body | authenticated |
| `204 No Content` (sessioninfo only) | **not** authenticated: a trap, it looks like success |
| `401 {"message":"Unauthenticated API request"}` | bad key or bad signature |
| `403 {"message":"Access denied."}` | signature fine, but this endpoint or scope is refused |

`/users/current` (mentioned in the docs intro, absent from the OpenAPI spec) answers a
valid API key with 403, so it isn't usable with API keys.

The key needs the **Read, Write and Delete** document scopes for this notebook (Write
for the §9 upload, Delete for its cleanup). Profile, purchase and share scopes are not
needed.

In [ ]:
r = os_get("users/sessioninfo")
show(r, "session info", limit=600)
ME = r.json() if r.status_code == 200 and r.text.strip() else {}
USER_ID = ME.get("id")
SCOPES = ME.get("oauth2Scopes")
FINDINGS["auth_ok"] = bool(USER_ID)
FINDINGS["key_scopes"] = SCOPES
FINDINGS["plan"] = {k: ME.get(k) for k in ("companyPlan", "planGroup", "role")}
if USER_ID:
    print("user id :", USER_ID)
    print("scopes  :", SCOPES)
    print("plan    :", FINDINGS["plan"])
elif r.status_code == 204:
    print("204: Onshape doesn't see an authenticated key. Check the key pair and that it "
          f"was made on this stack ({BASE_URL}).")
else:
    print(f"Authentication failed ({r.status_code}).")

### 3.2 What failure looks like (free: these are 4xx)

3DX taught us that failures can masquerade as success. Record what Onshape actually
returns for a **bad signature** and for a **skewed clock**, so the client can recognise
each later. `cache=False` so these always go live; they don't count toward the quota.

In [ ]:
bad_sig = os_request("GET", "documents", params={"limit": 1}, cache=False,
                     sign_kwargs={"secret": "definitely-not-the-secret"})
show(bad_sig, "bad signature", limit=400)

old_date = email.utils.formatdate(time.time() - 3600, usegmt=True)
skewed = os_request("GET", "documents", params={"limit": 1}, cache=False,
                    sign_kwargs={"date": old_date})
show(skewed, "clock skewed by 1 h", limit=400)

FINDINGS["bad_signature"] = (bad_sig.status_code, bad_sig.text[:200])
FINDINGS["clock_skew"] = (skewed.status_code, skewed.text[:200])

### 3.3 Company id

Webhooks (`companyId` scope) and revision endpoints need the company id. Educator
accounts should have one; whether they do is itself a finding.

In [ ]:
r = os_get("companies", params={"uid": USER_ID, "activeOnly": "true"})
show(r, "companies", limit=1000)
companies = (r.json().get("items") if r.ok else None) or []
COMPANY_ID = companies[0]["id"] if companies else None
FINDINGS["company"] = [(c.get("id"), c.get("name"), c.get("type")) for c in companies]
print("COMPANY_ID =", COMPANY_ID)

---
## 4. Scope: the "Inspection Parts" folder — *1 call*

The folder plays the role of the 3DX bookmark. What this section found on the first run:

- **Search doesn't find folders.** `GET /documents?q=<folder name>` returns no folders,
  so the folder is given by id. Paste either the id or the folder's whole URL
  (`…/documents?nodeId=<id>&resourceType=folder`) into `ONSHAPE_SCOPE_FOLDER`.
- **`GET /documents?parentId=<folder>` silently ignores `parentId`** (with or without
  `filter=0`). It returns 200 with *all* your documents, 20 per page. Each item does
  carry its own `parentId`, so client-side filtering works, at one call per 20
  documents.
- **`limit` is capped at 20** on `/documents`. `limit=50` gets a free
  `400 must be less than or equal to 20`.
- **`GET /globaltreenodes/folder/<id>` lists exactly the folder's contents** in one call,
  with the same document summaries (`modifiedAt`, `defaultWorkspace`, `parentId`). It's
  what the plan uses, but it is **not in the OpenAPI spec**, so the client keeps the
  client-side filter as a fallback.

In [ ]:
import re

m = re.search(r"[0-9a-f]{24}", SCOPE_FOLDER or "")
FOLDER_ID = m.group(0) if m else None
if not FOLDER_ID:
    print("Set ONSHAPE_SCOPE_FOLDER to the folder's id or URL: search by folder name "
          "returns nothing.")
print("FOLDER_ID =", FOLDER_ID)

In [ ]:
def list_folder(folder_id):
    # Primary: the folder-tree endpoint (undocumented, one call).
    r = os_get(f"globaltreenodes/folder/{folder_id}", params={"limit": 20})
    if r.ok:
        items, nxt = r.json().get("items", []), r.json().get("next")
        while nxt:                                  # >20 entries: follow pages
            page = os_get(nxt).json()
            items += page.get("items", [])
            nxt = page.get("next")
        return items, "globaltreenodes"
    # Fallback: page through every document and keep the folder's own.
    print(f"globaltreenodes answered {r.status_code}; falling back to client-side filter")
    items, url, params = [], "documents", {"limit": 20, "sortColumn": "modifiedAt",
                                           "sortOrder": "desc"}
    while url:
        page = os_get(url, params=params).json()
        items += [i for i in page.get("items", []) if i.get("parentId") == folder_id]
        url, params = page.get("next"), None
    return items, "documents+filter"

scope_docs, scope_subfolders = [], []
if FOLDER_ID:
    items, how = list_folder(FOLDER_ID)
    scope_docs = [i for i in items if i.get("resourceType") == "document"]
    scope_subfolders = [i for i in items if i.get("resourceType") == "folder"]
    FINDINGS["folder_listing_via"] = how
    FINDINGS["folder_lists_subfolders"] = bool(scope_subfolders)
    print(f"via {how}: {len(scope_docs)} document(s), {len(scope_subfolders)} sub-folder(s)")
    for d in scope_docs:
        print(f"doc  {d['id']}  modifiedAt={d.get('modifiedAt')}  {d.get('name')}")
    for f in scope_subfolders:
        print(f"dir  {f['id']}  {f.get('name')}")
    print(f"[ledger: {calls_used()}/{SPIKE_CALL_BUDGET}]")

> `modifiedAt` on each document is the cheap prefilter from plan §6: one call tells you
> which documents *might* have changed since the last sync.

---
## 5. One document: versions, elements, parts — *3 calls*

In [ ]:
DOC = scope_docs[0] if scope_docs else None
DID = DOC["id"] if DOC else None
WID = (DOC or {}).get("defaultWorkspace", {}).get("id")
print("DID =", DID, " WID =", WID, " name =", (DOC or {}).get("name"))

versions = []
if DID:
    r = os_get(f"documents/d/{DID}/versions")
    versions = r.json() if r.ok else []
    versions = versions if isinstance(versions, list) else versions.get("items", [])
    for v in versions:
        print(f"version {v['id']}  {v.get('name'):<10} created {v.get('createdAt')}  "
              f"microversion {v.get('microversion')}")
    print(f"[ledger: {calls_used()}/{SPIKE_CALL_BUDGET}]")
VID = versions[-1]["id"] if versions else None   # API returns oldest first; verify
print("latest VID =", VID)

In [ ]:
part_studios = []
if DID and VID:
    r = os_get(f"documents/d/{DID}/v/{VID}/elements")
    els = r.json() if r.ok else []
    for e in els:
        print(f"{e.get('elementType'):<12} {e['id']}  {e.get('name')}")
    part_studios = [e for e in els if e.get("elementType") == "PARTSTUDIO"]
EID = part_studios[0]["id"] if part_studios else None
print("EID =", EID)

### 5.1 Every part in the document, at that version, with thumbnails — one call

Leaving out the element returns parts from all Part Studios at once, which is the
batching plan §6.2 counts on. These fields become the catalog row.

In [ ]:
parts = []
if DID and VID:
    r = os_get(f"parts/d/{DID}/v/{VID}", params={"withThumbnails": "true"})
    parts = r.json() if r.ok else []
    for p in parts:
        print(f"{p.get('partId'):<6} {p.get('name'):<20} elem={p.get('elementId')} "
              f"pn={p.get('partNumber')!r} rev={p.get('revision')!r} "
              f"state={p.get('state')} mv={p.get('microversionId')}")
    if parts:
        print("\nOne part, all fields:")
        brief(parts[0], limit=2500)
    print(f"[ledger: {calls_used()}/{SPIKE_CALL_BUDGET}]")
FINDINGS["part_fields"] = sorted(parts[0].keys()) if parts else None

---
## 6. The change-stamp experiment — *3 calls per version, once*

**The most important verify item.** Plans live in the design document, so the catalog's
`cestamp` must be per-part: it must change when a part's geometry changes and *not*
when something else in the document changes. With the V1/V2/V3 setup from the top:

| Transition | Wanted |
|---|---|
| V1 → V2 (only A edited) | A changes, B unchanged |
| V2 → V3 (no change to the parts) | A and B unchanged |

**Result of the first run (2026-10-16):**

- **Part `microversionId` is the document's microversion** at that version, identical
  to the version's own `microversion`. Both parts changed on every transition, so it is
  useless as a per-part stamp. `metadataMicroversion` and the per-version part `id`
  also change every time.
- **Element `microversionId`** (from the elements list) is per tab. The Assembly's
  stayed `35bb…` throughout while the Part Studio's changed on each step, so it is a
  good *prefilter*: unchanged → no part in that Part Studio changed.
- **Mass properties** (`GET /partstudios/…/massproperties?partId=…`, all parts in one
  call) changed only for part A on V1→V2 and for nothing on V2→V3. That is the
  per-part stamp: a hash of rounded volume, centroid and inertia.

The cell below reproduces all three columns.

In [ ]:
import hashlib as _hl

def mass_fingerprint(body, digits=9):
    # volume/centroid/inertia come as [value, min, max] (or 3x3 / xyz arrays of those);
    # keep the values, rounded, and hash them. Mass is 0 without a material, so skip it.
    def vals(x):
        return [round(float(v), digits) for v in (x or [])]
    key = json.dumps([vals(body.get("volume"))[:1], vals(body.get("centroid")),
                      vals(body.get("inertia"))])
    return _hl.sha1(key.encode()).hexdigest()[:10]

rows = []
for v in versions[-3:]:
    vid, vname = v["id"], v["name"]
    els = os_get(f"documents/d/{DID}/v/{vid}/elements").json()
    el_mv = {e["id"]: e.get("microversionId") for e in els}
    pts = {p["partId"]: p for p in os_get(f"parts/d/{DID}/v/{vid}").json()}
    mp = os_get(f"partstudios/d/{DID}/v/{vid}/e/{EID}/massproperties",
                params={"partId": sorted(pts), "massAsGroup": "false"}).json().get("bodies", {})
    for pid, p in sorted(pts.items()):
        rows.append({"version": vname, "part": pid, "name": p.get("name"),
                     "part_mv": (p.get("microversionId") or "")[:8],
                     "element_mv": (el_mv.get(p.get("elementId")) or "")[:8],
                     "mass_fp": mass_fingerprint(mp.get(pid, {}))})

print(f"{'ver':<5}{'part':<6}{'part_mv':<10}{'element_mv':<12}{'mass_fp':<12}name")
for r_ in rows:
    print(f"{r_['version']:<5}{r_['part']:<6}{r_['part_mv']:<10}{r_['element_mv']:<12}"
          f"{r_['mass_fp']:<12}{r_['name']}")

def changed(field):
    out = []
    names = [v["name"] for v in versions[-3:]]
    for a, b in zip(names, names[1:]):
        fa = {r_["part"]: r_[field] for r_ in rows if r_["version"] == a}
        fb = {r_["part"]: r_[field] for r_ in rows if r_["version"] == b}
        out.append((a, b, sorted(p for p in fa if fa[p] != fb.get(p))))
    return out

for field in ("part_mv", "element_mv", "mass_fp"):
    print(f"{field:<11} changed:", changed(field))
FINDINGS["stamp_part_microversion"] = changed("part_mv")
FINDINGS["stamp_element_microversion"] = changed("element_mv")
FINDINGS["stamp_mass_fingerprint"] = changed("mass_fp")
print(f"[ledger: {calls_used()}/{SPIKE_CALL_BUDGET}]")

---
## 7. Thumbnails — *1 call*

Unlike our 3DX tenant, Onshape has real images. `skipDefaultImage=true` asks for a 404
rather than a placeholder when none exists. Binary, so it's saved to disk rather than the
JSON cache; re-running reuses the file.

In [ ]:
thumb_path = SPIKE_DIR / "files" / f"thumb_{DID}_{VID}_{EID}.png"
if EID and not thumb_path.exists():
    r = os_get(f"thumbnails/d/{DID}/v/{VID}/e/{EID}/s/300x300",
               params={"skipDefaultImage": "true"}, accept="image/png", cache=False)
    print(r.status_code, r.headers.get("Content-Type"), len(r.content), "bytes")
    if r.ok and r.headers.get("Content-Type", "").startswith("image"):
        thumb_path.write_bytes(r.content)
if thumb_path.exists():
    from IPython.display import Image, display
    display(Image(filename=str(thumb_path)))
FINDINGS["element_thumbnail"] = thumb_path.exists()

---
## 8. STEP export of one part — *about 4–8 calls, once*

The 3DX `dsdo` derived output, Onshape style: ask for a translation, poll it, download
the result. Verify items settled here:

- Does the generic `/translations` endpoint honour `partIds` (one part only)?
- Which unit field works, and does the file come out in metres?
- Does `externaldata` return raw STEP or a zip?

`storeInDocument: false` keeps the result out of the design document. The final file
is cached by (document, version, element, part), so this whole section runs live only
once.

In [ ]:
PART = parts[0] if parts else None
PID = PART["partId"] if PART else None
EID_PART = PART.get("elementId", EID) if PART else EID

r = os_get("translations/translationformats")
formats = r.json() if r.ok else []
step_fmt = [f for f in formats if "STEP" in (f.get("name") or "").upper()]
brief([{k: f.get(k) for k in ("name", "translatorName", "validDestinationFormat",
                              "validSourceFormat")} for f in step_fmt])
STEP_FORMAT_NAME = step_fmt[0]["name"] if step_fmt else "STEP"

In [ ]:
import io
import zipfile

step_path = SPIKE_DIR / "files" / f"{DID}_{VID}_{EID_PART}_{PID}.step"

def export_part_step(did, vid, eid, pid, dest):
    body = {
        "formatName": STEP_FORMAT_NAME,
        "partIds": pid,
        "stepVersionString": "AP242",
        "unit": "METER",          # verify: honoured? (compare bbox below)
        "storeInDocument": False,
    }
    r = os_request("POST", f"partstudios/d/{did}/v/{vid}/e/{eid}/translations",
                   json_body=body)
    show(r, "create translation", limit=800)
    r.raise_for_status()
    tid = r.json()["id"]

    delay = 2.0
    for attempt in range(8):          # ~2+3+4.5+... s; well under 8 polls in practice
        time.sleep(delay)
        st = os_request("GET", f"translations/{tid}", cache=False).json()
        print(f"poll {attempt + 1}: {st.get('requestState')}")
        if st.get("requestState") in ("DONE", "FAILED"):
            break
        delay = min(delay * 1.5, 15)
    FINDINGS["translation_final"] = {k: st.get(k) for k in
                                     ("requestState", "failureReason",
                                      "resultExternalDataIds", "resultElementIds")}
    if st.get("requestState") != "DONE":
        raise RuntimeError(f"translation ended {st.get('requestState')}: {st.get('failureReason')}")

    fid = st["resultExternalDataIds"][0]
    dl = os_request("GET", f"documents/d/{did}/externaldata/{fid}",
                    accept="application/octet-stream", cache=False)
    dl.raise_for_status()
    raw = dl.content
    is_zip = raw[:2] == b"PK"
    FINDINGS["externaldata_is_zip"] = is_zip
    if is_zip:
        with zipfile.ZipFile(io.BytesIO(raw)) as z:
            names = z.namelist()
            FINDINGS["zip_members"] = names
            raw = z.read(next(n for n in names if n.lower().endswith((".step", ".stp"))))
    dest.write_bytes(raw)
    return dest

if PID and not step_path.exists():
    export_part_step(DID, VID, EID_PART, PID, step_path)
if step_path.exists():
    head = step_path.read_text(errors="replace")[:2000]
    print(step_path, step_path.stat().st_size, "bytes")
    print("\n".join(l for l in head.splitlines() if l.startswith(("ISO-10303", "FILE_SCHEMA", "FILE_NAME")))[:600])
    units = re.findall(r"SI_UNIT\s*\(\s*([^,]*),\s*\.METRE\.\)", step_path.read_text(errors="replace"))
    print("length unit prefixes in file:", units or "(none found)")
    FINDINGS["step_length_unit"] = units
    FINDINGS["step_bodies"] = step_path.read_text(errors="replace").count("MANIFOLD_SOLID_BREP")
    print("MANIFOLD_SOLID_BREP count:", FINDINGS["step_bodies"], "(1 means partIds was honoured)")
print(f"[ledger: {calls_used()}/{SPIKE_CALL_BUDGET}]")

### 8.1 Does it load, and at the right scale? — *1 call*

Compare the loader's bounding box (metres) with Onshape's own bounding box for the part
(also metres). This checks the unit handling end to end, whatever the file declares.

> **First run (2026-10-16):** the export honours `partIds` (1 solid), returns raw STEP
> (not a zip), and declares plain metres (`SI_UNIT($,.METRE.)`). But the loader in the
> current image returned **43.1 × 52.9 × 200** against Onshape's **0.043 × 0.053 × 0.2 m**:
> OpenCascade converts every STEP to millimetres on read, and `load_step(units="m")`
> doesn't scale. Separately, the image's cadquery-ocp 8.0.1 broke the loader's imports.
> Both are loader bugs, not Onshape ones; see `ONSHAPE_API_FINDINGS.md`.

`step_loader` needs OCP and Open3D, which live in the container, not on the host. If the
import fails here, the cell prints a `docker exec` command that runs the same check in
the container. The STEP file is already there: `./catalog` is mounted at `/catalog`.

In [ ]:
import sys
sys.path.insert(0, str(REPO_ROOT / "ros2_ws/src/ViewpointGeneration/viewpoint_generation"))

bb_r = os_get(f"parts/d/{DID}/v/{VID}/e/{EID_PART}/partid/{PID}/boundingboxes") if PID else None
if bb_r is not None and bb_r.ok:
    bb = bb_r.json()
    onshape_size = [bb["highX"] - bb["lowX"], bb["highY"] - bb["lowY"], bb["highZ"] - bb["lowZ"]]
    print("Onshape bbox size (m):", [round(x, 6) for x in onshape_size])

try:
    if not step_path.exists():
        raise FileNotFoundError(f"{step_path} not downloaded yet (see §8)")
    from viewpoint_generation.step_loader import load_step
    # STEP files carry their own units and OCC converts on read; 'm' is our target unit.
    result = load_step(step_path, units="m")
    ext = result.mesh.get_axis_aligned_bounding_box().get_extent()
    print("Loader bbox size (m): ", [round(float(x), 6) for x in ext])
    if bb_r is not None and bb_r.ok:
        ratio = max(ext) / max(onshape_size)
        print(f"loader/Onshape size ratio: {ratio:.4g}"
              + ("  <-- 1000x: loader didn't convert OCC's millimetres to metres"
                 if abs(ratio - 1000) < 1 else ""))
        FINDINGS["loader_scale_ratio"] = ratio
    print("B-rep faces:", result.face_count(), result.surface_type_counts())
    FINDINGS["loader_bbox_m"] = [float(x) for x in ext]
    FINDINGS["onshape_bbox_m"] = onshape_size
except ImportError as e:
    in_container = "/catalog/" + str(step_path.relative_to(CATALOG_ROOT))
    print(f"step_loader unavailable on this machine ({e}). Run in the container:\n")
    print(f"docker exec -it ${{CONTAINER_NAME:-ros2-docker-template}} bash -ic \"python3 -c '"
          f"from viewpoint_generation.step_loader import load_step; "
          f"r = load_step(\\\"{in_container}\\\", units=\\\"m\\\"); "
          f"print(list(r.mesh.get_axis_aligned_bounding_box().get_extent()), "
          f"r.face_count(), r.surface_type_counts())'\"")
except FileNotFoundError as e:
    print(e)

---
## 9. Writes: a plan tab in the design document — *about 4 calls, `ALLOW_WRITES` only*

The plan §8 storage: upload a small JSON as a **blob element** in the document's
workspace, find it again by name, download it, then delete it. Also tests whether
upload can target a **tab folder** (`locationGroupId` is in the upload schema; verify).

This edits the workspace (a new microversion), which is exactly what plan §8 says must
*not* change any part's stamp. Re-run §6 after making a new version to confirm.

In [ ]:
def writes_enabled(what):
    if not ALLOW_WRITES:
        print(f"Skipped ({what}): set ALLOW_WRITES = True in §1 to run.")
    return ALLOW_WRITES

PLAN_TAB = None
if DID and WID and writes_enabled("upload plan blob"):
    payload = {"spike": True, "created": dt.datetime.now(dt.timezone.utc).isoformat(),
               "plm_context": {"source": "onshape", "did": DID, "vid": VID, "pid": PID}}
    name = f"PLAN_SPIKE_{PID}_V{len(versions)}_ORDERED.json"
    r = os_request("POST", f"blobelements/d/{DID}/w/{WID}",
                   files={"file": (name, json.dumps(payload).encode(), "application/json")},
                   data={"encodedFilename": name})
    show(r, "upload blob", limit=800)
    if r.ok:
        PLAN_TAB = r.json().get("id")
print("PLAN_TAB =", PLAN_TAB)

In [ ]:
if PLAN_TAB:
    r = os_request("GET", f"documents/d/{DID}/w/{WID}/elements", cache=False)
    tabs = [e for e in (r.json() if r.ok else []) if e["id"] == PLAN_TAB]
    brief(tabs)
    FINDINGS["blob_tab"] = tabs[0] if tabs else None

    dl = os_request("GET", f"blobelements/d/{DID}/w/{WID}/e/{PLAN_TAB}",
                    accept="application/octet-stream", cache=False)
    print("download:", dl.status_code, dl.content[:200])
    FINDINGS["blob_roundtrip"] = dl.ok and b'"spike": true' in dl.content

In [ ]:
# Clean up: delete the spike tab. Comment this out to keep it for the §6 re-check.
if PLAN_TAB and writes_enabled("delete plan blob"):
    r = os_request("DELETE", f"elements/d/{DID}/w/{WID}/e/{PLAN_TAB}")
    print("delete:", r.status_code, r.text[:200])

---
## 10. Webhooks: permission check only — *1 call*

Registration waits for the relay (plan §6.1, Phase 5). This only checks the endpoint is
reachable with these keys on an Educator plan, and lists anything already registered.

In [ ]:
r = os_get("webhooks", params={"company": COMPANY_ID} if COMPANY_ID else {"user": USER_ID},
           cache=False)
show(r, "webhooks", limit=800)
FINDINGS["webhooks_list"] = (r.status_code, len(r.json().get("items", [])) if r.ok else None)

---
## 11. Findings and cost

Copy this into `ONSHAPE_API_FINDINGS.md` (the counterpart of
`../3dx_integration/TENANT_API_FINDINGS.md`) and tick the plan's §13 verify list.

In [ ]:
brief(FINDINGS, limit=10000)

with LEDGER.open() as f:
    rows = [json.loads(l) for l in f]
billable = [r for r in rows if r["billable"]]
print(f"\nLive calls: {len(rows)} total, {len(billable)} billable, "
      f"{len(rows) - len(billable)} free (4xx/5xx).")
by_path = {}
for r in billable:
    key = re.sub(r"[0-9a-f]{24}", "{id}", r["path"])
    by_path[key] = by_path.get(key, 0) + 1
for k, v in sorted(by_path.items(), key=lambda kv: -kv[1]):
    print(f"{v:4d}  {k}")
(SPIKE_DIR / "findings.json").write_text(json.dumps(FINDINGS, indent=2, default=str))
print("\nSaved", SPIKE_DIR / "findings.json")